## MCP Agent with Microsoft Agent Framework

### Installing Utilities and Libraries

In [ ]:
%pip install agent-framework==1.19.0 azure-ai-projects==2.7.0 azure-identity==1.25.3 azure-monitor-opentelemetry==1.8.8

### Setting up the Environment Variables

In [ ]:
import os
from dotenv import load_dotenv
from azure.identity import AzureCliCredential

load_dotenv()
foundry_project_endpoint = os.getenv("FOUNDRY_PROJECT_ENDPOINT")
model_deployment_name = os.getenv("MODEL_DEPLOYMENT_NAME")
mcp_server_name = os.getenv("MCP_SERVER_NAME")

credential = AzureCliCredential()

### Creating the Foundry AI Project Client

In [ ]:
from agent_framework import Agent
from azure.ai.projects import AIProjectClient
from agent_framework.foundry import FoundryAgent, FoundryChatClient, to_prompt_agent
from azure.identity import AzureCliCredential

foundry_project_client = AIProjectClient(
    endpoint = foundry_project_endpoint,
    credential = credential
)

### Getting the MCP Server Connection ID

In [ ]:
connection_id = ""

for connection in foundry_project_client.connections.list():
    if connection.name == mcp_server_name:
        connection_id = connection.id
        break

print(f"The MCP Server Connection ID is: {connection_id}")

### Creating the MCP Tool Spec

In [ ]:
from azure.ai.projects.models import MCPTool

tool = MCPTool(
    server_label = "microsoft_learn_server",
    server_url="https://learn.microsoft.com/api/mcp",
    require_approval="never",
    project_connection_id=connection_id
)

### Creating the MCP Agent

In [ ]:
from azure.ai.projects.models import PromptAgentDefinition

agent_name = "MAF-MCP-Agent"

agent = foundry_project_client.agents.create_version(
    agent_name=agent_name,
    definition=PromptAgentDefinition(
        model=model_deployment_name,
        instructions="You are an intelligent assistant that can interact with the Microsoft Learn MCP server to provide users with relevant learning resources and information about Microsoft technologies.",
        tools=[tool],
    )
)

print(f"Created MCP Agent with ID: {agent.id}")

### Create a MAF Agent

In [ ]:
foundry_agent = FoundryAgent(
    project_endpoint=foundry_project_endpoint,
    agent_name=agent.name,
    agent_version=agent.version,
    credential=credential
)

### Create a Session to Maintain Conversation History

In [ ]:
session = foundry_agent.create_session()

### Request Streaming Responses from the Agent

In [ ]:
user_query = """Can you please help me prepare for the AI-500: Multi-Agent AI Solutions Exam 
                in a month's worth of time? Use the MSLearn MCP Server for latest info"""

In [ ]:
print("Agent (streaming): ", end="", flush=True)
async for chunk in foundry_agent.run(user_query, session=session, stream=True):
        if chunk.text:
            print(chunk.text, end="", flush=True)
print()